# Effect of confound regression on graph-measure variability

## Combined population, Parkinson's disease, and healthy controls

This notebook compares graph-measure variability between the **no-confound-regression** and **with-confound-regression** pipelines. It generates six result figures—local and global metrics for PD+HC, PD, and HC—and three permutation-null diagnostic figures.

The visual system matches `Fig2and3.ipynb`: numerical variability (NV) is light gray, population variability (PV) is dark gray, and NPVR uses black for PD+HC, red for PD, and royal blue for HC. Stars denote mean NPVR and dashed lines connect threshold-wise means.

## Reproducibility

Python 3.11+ dependencies:

```bash
python -m pip install "numpy>=2" "pandas>=2" "matplotlib>=3.7" "scipy>=1.11"
```

Set `FMRI_DATA_ROOT` to the directory containing `Allpop/`. The notebook discovers the original project layout automatically. Every plotting call generates a PDF directly in both `build/S2/figures/` and `notebooksfogures/Figures/`, plus a PNG preview for GitHub.

In [ ]:
from __future__ import annotations

import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from scipy.stats import permutation_test


THRESHOLDS = (0.05, 0.1, 0.2, 0.3, 0.4, 0.5)
THRESHOLD_LABELS = {0.05: "05", 0.1: "1", 0.2: "2", 0.3: "3", 0.4: "4", 0.5: "5"}

LOCAL_METRICS = {
    "degree": "Degree Centrality",
    "betweenness": "Betweenness Centrality",
    "eigenvector": "Eigenvector Centrality",
    "clustering": "Clustering Coefficient",
}
GLOBAL_METRICS = {
    "small_worldness": "Small-worldness",
    "average_shortest_path_length": "Average Shortest Path Length",
}
METRICS = {**LOCAL_METRICS, **GLOBAL_METRICS}

CSV_COLUMNS = {
    "degree": "degree",
    "betweenness": "betweeness",
    "eigenvector": "eigenvec",
    "clustering": "clusteringcoef",
    "small_worldness": "smallworldness",
    "average_shortest_path_length": "avg_shortestPathLength",
}
FRAME_TOKENS = CSV_COLUMNS.copy()

COLORS = {
    "numerical": "#B6B1B1",
    "population": "#353333",
    "combined": "black",
    "pd": "red",
    "hc": "royalblue",
}
GROUP_LABELS = {"combined": "PD+HC", "pd": "PD", "hc": "HC"}


def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


def find_data_root(repository_root: Path) -> Path:
    if configured := os.environ.get("FMRI_DATA_ROOT"):
        candidate = Path(configured).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"FMRI_DATA_ROOT does not exist: {candidate}")
        return candidate
    for candidate in (repository_root, *repository_root.parents):
        if (candidate / "Allpop").is_dir():
            return candidate
    raise FileNotFoundError("Set FMRI_DATA_ROOT to the directory containing Allpop/.")


REPOSITORY_ROOT = find_repository_root()
DATA_ROOT = find_data_root(REPOSITORY_ROOT)
ALLPOP_DIR = DATA_ROOT / "Allpop"
VARIABILITY_DIR = ALLPOP_DIR / "pickles" / "pdhc"
DIFFERENCE_DATA = REPOSITORY_ROOT / "notebooks" / "difference_data.pkl"
PAPER_FIGURE_DIR = REPOSITORY_ROOT / "build" / "S2" / "figures"
PUBLIC_FIGURE_DIR = REPOSITORY_ROOT / "notebooksfogures" / "Figures"
PAPER_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PUBLIC_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data:          {ALLPOP_DIR}")
print(f"Paper PDFs:    {PAPER_FIGURE_DIR}")
print(f"Public output: {PUBLIC_FIGURE_DIR}")

## 1. Load and aggregate subject-level NPVR

The CSV files store squared regional or global terms. For each population and threshold, the group estimate is $\sqrt{|\operatorname{mean}(x)|}$, matching the original workflow.

In [ ]:
def parse_array(value: str) -> np.ndarray:
    """Parse a NumPy-style array serialized in a CSV cell."""
    return np.fromstring(str(value).replace("[", " ").replace("]", " "), sep=" ")


def aggregate_subject_npvr(path: Path) -> dict[str, np.ndarray | float]:
    """Aggregate subject-level squared terms into group NPVR estimates."""
    frame = pd.read_csv(path)
    result = {}
    for metric in LOCAL_METRICS:
        arrays = frame[CSV_COLUMNS[metric]].map(parse_array).to_list()
        lengths = {array.size for array in arrays}
        if len(lengths) != 1:
            raise ValueError(f"Inconsistent regional arrays in {path.name}")
        result[metric] = np.sqrt(np.abs(np.mean(np.stack(arrays), axis=0)))
    for metric in GLOBAL_METRICS:
        result[metric] = float(np.sqrt(abs(frame[CSV_COLUMNS[metric]].mean())))
    return result


def subject_npvr_path(pipeline: str, group: str, threshold: float) -> Path:
    prefixes = {
        ("with", "pd"): "dfW_stat", ("with", "hc"): "dfWhc_stat",
        ("no", "pd"): "dfN_stat", ("no", "hc"): "dfNhc_stat",
    }
    return ALLPOP_DIR / f"{prefixes[(pipeline, group)]}{THRESHOLD_LABELS[threshold]}.csv"


group_npvr = {
    pipeline: {
        group: {
            threshold: aggregate_subject_npvr(subject_npvr_path(pipeline, group, threshold))
            for threshold in THRESHOLDS
        }
        for group in ("pd", "hc")
    }
    for pipeline in ("with", "no")
}

sample_sizes = pd.DataFrame(
    {
        "threshold": threshold,
        "PD": len(pd.read_csv(subject_npvr_path("with", "pd", threshold))),
        "HC": len(pd.read_csv(subject_npvr_path("with", "hc", threshold))),
    }
    for threshold in THRESHOLDS
).set_index("threshold")
sample_sizes

## 2. Compute pipeline differences

To preserve the paper analysis exactly, the original sign conventions are retained:

$$\Delta\mathrm{NPVR}=\mathrm{NPVR}_{\mathrm{no\ confound}}-\mathrm{NPVR}_{\mathrm{with\ confound}},$$

$$\Delta\mathrm{NV}=\mathrm{NV}_{\mathrm{with\ confound}}-\mathrm{NV}_{\mathrm{no\ confound}},\qquad
\Delta\mathrm{PV}=\mathrm{PV}_{\mathrm{with\ confound}}-\mathrm{PV}_{\mathrm{no\ confound}}.$$

The combined-population differences come from the pipeline's saved `difference_data.pkl`; PD and HC differences are reconstructed from their group files.

In [ ]:
def subtract_metrics(first: dict, second: dict) -> dict:
    return {
        metric: np.asarray(first[metric], dtype=float) - np.asarray(second[metric], dtype=float)
        for metric in METRICS
    }


npvr_difference = {
    group: {
        threshold: subtract_metrics(
            group_npvr["no"][group][threshold],
            group_npvr["with"][group][threshold],
        )
        for threshold in THRESHOLDS
    }
    for group in ("pd", "hc")
}


def unwrap_numeric(value) -> np.ndarray | float:
    while isinstance(value, (pd.Series, pd.DataFrame)):
        value = value.iloc[0] if isinstance(value, pd.Series) else value.iloc[0, 0]
    array = np.asarray(value, dtype=float).squeeze()
    return float(array) if array.ndim == 0 else array.reshape(-1)


with DIFFERENCE_DATA.open("rb") as stream:
    combined_saved = pickle.load(stream)

COMBINED_LOCAL_KEYS = {
    "degree": "degree_ratio", "betweenness": "betweeness_ratio",
    "eigenvector": "eigenvec_ratio", "clustering": "clusteringcoef_ratio",
}
COMBINED_GLOBAL_KEYS = {
    "small_worldness": "ratio_smallworldness",
    "average_shortest_path_length": "ratio_avg_shortestPathLength",
}

npvr_difference["combined"] = {}
for threshold in THRESHOLDS:
    label = THRESHOLD_LABELS[threshold]
    local_frame = combined_saved[f"diff_ratio{label}"]
    global_values = combined_saved[f"diff_avgratio{label}G"]
    values = {
        metric: unwrap_numeric(local_frame[column].iloc[0])
        for metric, column in COMBINED_LOCAL_KEYS.items()
    }
    values.update(
        {
            metric: unwrap_numeric(global_values[column])
            for metric, column in COMBINED_GLOBAL_KEYS.items()
        }
    )
    npvr_difference["combined"][threshold] = values

In [ ]:
def frame_metric(frame: pd.DataFrame, metric: str) -> np.ndarray | float:
    matches = [column for column in frame.columns if FRAME_TOKENS[metric] in column]
    if not matches:
        raise KeyError(f"Could not find {metric!r} in {frame.columns.tolist()}")
    return unwrap_numeric(frame[matches[0]].iloc[0])


def variability_path(group: str, pipeline: str, threshold: float, kind: str) -> Path:
    group_prefix = group if pipeline == "with" else f"{group}N"
    return VARIABILITY_DIR / (
        f"{group_prefix}_avrg{THRESHOLD_LABELS[threshold]}_Wconf_{kind}.pkl"
    )


variability_difference = {group: {} for group in ("pd", "hc")}
for group in variability_difference:
    for threshold in THRESHOLDS:
        variability_difference[group][threshold] = {}
        for result_name, file_kind in (("numerical", "num"), ("population", "anat")):
            with_frame = pd.read_pickle(variability_path(group, "with", threshold, file_kind))
            no_frame = pd.read_pickle(variability_path(group, "no", threshold, file_kind))
            variability_difference[group][threshold][result_name] = {
                metric: np.asarray(frame_metric(with_frame, metric), dtype=float)
                - np.asarray(frame_metric(no_frame, metric), dtype=float)
                for metric in METRICS
            }


variability_difference["combined"] = {}
for threshold in THRESHOLDS:
    label = THRESHOLD_LABELS[threshold]
    variability_difference["combined"][threshold] = {
        result_name: {
            metric: np.asarray(frame_metric(combined_saved[f"diff_{saved_name}{label}"], metric), dtype=float)
            for metric in METRICS
        }
        for result_name, saved_name in (("numerical", "num"), ("population", "anat"))
    }

print("Computed pooled PD+HC, PD, and HC NV, PV, and NPVR differences at six thresholds.")

## 3. One-sample permutation inference

For every local metric, threshold, and population, a one-sample sign-flip permutation test evaluates whether the regional mean difference is below zero. The 24 tests within each population are Bonferroni-corrected. A fixed random seed reproduces the original analysis.

In [ ]:
def run_permutation_analysis(group: str) -> dict:
    raw_p_values, results = [], {}
    for threshold in THRESHOLDS:
        results[threshold] = {}
        for metric in LOCAL_METRICS:
            values = np.asarray(npvr_difference[group][threshold][metric], dtype=float)
            result = permutation_test(
                (values,), np.mean, permutation_type="samples", vectorized=True,
                alternative="less", random_state=1, axis=0,
            )
            raw_p_values.append(float(result.pvalue))
            results[threshold][metric] = {
                "statistic": float(result.statistic),
                "null": np.asarray(result.null_distribution, dtype=float),
            }

    corrected = np.minimum(np.asarray(raw_p_values) * len(raw_p_values), 1.0)
    index = 0
    for threshold in THRESHOLDS:
        for metric in LOCAL_METRICS:
            results[threshold][metric]["p_corrected"] = float(corrected[index])
            index += 1
    return results


permutation_results = {
    group: run_permutation_analysis(group) for group in ("combined", "pd", "hc")
}

permutation_table = pd.DataFrame(
    {
        "population": GROUP_LABELS[group], "threshold": threshold,
        "metric": LOCAL_METRICS[metric],
        "mean_difference": result["statistic"],
        "p_bonferroni": result["p_corrected"],
    }
    for group, group_results in permutation_results.items()
    for threshold, threshold_results in group_results.items()
    for metric, result in threshold_results.items()
)
permutation_table.round(4)

## 4. Figure style and export helpers

All result figures use the same dimensions, metric order, axis language, gray NV/PV palette, group colors, threshold labels, and bottom captions as `Fig2and3.ipynb`.

In [ ]:
def finite_values(value) -> np.ndarray:
    values = np.asarray(value, dtype=float).reshape(-1)
    return values[np.isfinite(values)]


def regional_mean(value) -> float:
    return float(np.nanmean(finite_values(value)))


def significance_label(p_value: float) -> str:
    if p_value < 0.001: return "***"
    if p_value < 0.01: return "**"
    if p_value < 0.05: return "*"
    return "ns"


def draw_box(axis, values, position, color, width=0.20) -> None:
    values = finite_values(values)
    parts = axis.boxplot(
        [values], positions=[position], widths=width, patch_artist=True,
        showfliers=False, manage_ticks=False,
    )
    parts["boxes"][0].set(facecolor=to_rgba(color, 0.35), edgecolor=color, linewidth=1.2)
    for element in ("whiskers", "caps", "medians"):
        plt.setp(parts[element], color=color, linewidth=1.0)
    jitter = np.linspace(-0.035, 0.035, len(values))
    axis.scatter(position + jitter, values, s=7, color=color, alpha=0.45, edgecolors="none")


def save_figure(figure: plt.Figure, stem: str) -> None:
    """Generate PDFs directly in both output directories and a PNG preview."""
    for directory in (PAPER_FIGURE_DIR, PUBLIC_FIGURE_DIR):
        path = directory / f"{stem}.pdf"
        figure.savefig(path, format="pdf", dpi=300, bbox_inches="tight")
        print(f"Generated PDF: {path}")
    figure.savefig(PUBLIC_FIGURE_DIR / f"{stem}.png", dpi=180, bbox_inches="tight")


def add_combined_caption(figure: plt.Figure, y=0.01) -> None:
    handles = [
        Line2D([0], [0], color=COLORS[group], marker="*", linestyle="--",
               markersize=10, label=f"{GROUP_LABELS[group]}: Mean NPVR difference")
        for group in ("combined", "pd", "hc")
    ]
    figure.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, y),
                  ncol=3, frameon=False)


def add_group_caption(figure: plt.Figure, group: str, y=0.01,
                      numerical_color: str | None = None,
                      population_color: str | None = None) -> None:
    numerical_color = numerical_color or COLORS["numerical"]
    population_color = population_color or COLORS["population"]
    handles = [
        Patch(facecolor="none", edgecolor="none", label=f"Group: {GROUP_LABELS[group]}"),
        Patch(facecolor=to_rgba(numerical_color, 0.35), edgecolor=numerical_color,
              label=r"NV difference ($\Delta\sigma_{num}$)"),
        Patch(facecolor=to_rgba(population_color, 0.35), edgecolor=population_color,
              label=r"PV difference ($\Delta\sigma_{pop}$)"),
        Patch(facecolor=to_rgba(COLORS[group], 0.35), edgecolor=COLORS[group],
              label=r"NPVR difference ($\Delta$NPVR)"),
        Line2D([0], [0], color=COLORS[group], marker="*", linestyle="--",
               markersize=10, markeredgecolor="black", label="Mean NPVR difference"),
    ]
    figure.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, y),
                  ncol=5, frameon=False)

## 5. Combined PD+HC, PD, and HC NPVR differences

Regional distributions are shown for local metrics; global metrics are scalar threshold-wise trends.

In [ ]:
def plot_combined_local() -> plt.Figure:
    figure, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
    x = np.arange(len(THRESHOLDS), dtype=float)
    offsets = {"combined": -0.24, "pd": 0.0, "hc": 0.24}

    for axis, (metric, label) in zip(axes, LOCAL_METRICS.items()):
        all_values = [finite_values(npvr_difference[g][t][metric]) for g in offsets for t in THRESHOLDS]
        combined_values = np.concatenate(all_values)
        limit = np.percentile(np.abs(combined_values), 97.5) * 1.20
        if limit == 0:
            limit = 1.0
        axis.set_ylim(-limit, limit)
        for group, offset in offsets.items():
            means = []
            for index, threshold in enumerate(THRESHOLDS):
                values = npvr_difference[group][threshold][metric]
                position = x[index] + offset
                draw_box(axis, values, position, COLORS[group], width=0.18)
                mean = regional_mean(values)
                means.append(mean)
                axis.plot(position, mean, marker="*", markersize=11, color=COLORS[group],
                          markeredgecolor="black", linestyle="None")
                p_value = permutation_results[group][threshold][metric]["p_corrected"]
                axis.text(position, min(max(finite_values(values)) + 0.04 * limit, 0.92 * limit),
                          significance_label(p_value), ha="center", color=COLORS[group], fontsize=11)
            axis.plot(x + offset, means, color=COLORS[group], linestyle="--", linewidth=1.8)
        axis.axhline(0, color="gray", linewidth=0.8)
        axis.set_title(label, fontweight="bold", fontsize=16)
        axis.set_ylabel("NPVR difference", fontsize=14)
        axis.tick_params(axis="both", labelsize=14)
        axis.grid(axis="y", color="0.9", linewidth=0.8)

    axes[-1].set_xticks(x, [f"T = {t:g}" for t in THRESHOLDS])
    axes[-1].set_xlabel("Threshold Values", fontweight="bold", fontsize=14)
    axes[-1].text(0.5, -0.19, "* p<0.05; ** p<0.01; *** p<0.001; ns = not significant",
                  transform=axes[-1].transAxes, ha="center", fontsize=10)
    add_combined_caption(figure, 0.005)
    figure.tight_layout(rect=(0, 0.08, 1, 1))
    return figure


def plot_combined_global() -> plt.Figure:
    figure, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    x = np.arange(len(THRESHOLDS), dtype=float)
    for axis, (metric, label) in zip(axes, GLOBAL_METRICS.items()):
        for group in ("combined", "pd", "hc"):
            values = [float(npvr_difference[group][t][metric]) for t in THRESHOLDS]
            axis.plot(x, values, color=COLORS[group], marker="*", markersize=11,
                      linestyle="--", linewidth=1.8, markeredgecolor="black")
        axis.axhline(0, color="gray", linewidth=0.8)
        axis.set_title(label, fontweight="bold", fontsize=16)
        axis.set_ylabel("NPVR difference", fontsize=14)
        axis.tick_params(axis="both", labelsize=14)
        axis.grid(axis="y", color="0.9", linewidth=0.8)
    axes[-1].set_xticks(x, [f"T = {t:g}" for t in THRESHOLDS])
    axes[-1].set_xlabel("Threshold", fontweight="bold", fontsize=14)
    add_combined_caption(figure, 0.005)
    figure.tight_layout(rect=(0, 0.10, 1, 1))
    return figure


combined_local = plot_combined_local()
save_figure(combined_local, "fig5-localPD+HC")
plt.show()

combined_global = plot_combined_global()
save_figure(combined_global, "fig5-globalPD+HC")
plt.show()

### Additional local figure — mean NPVR stars only

This version removes the regional NPVR boxplots and points. Each star is the regional mean NPVR difference; dashed lines connect thresholds and the colored label above each star reports Bonferroni-corrected significance.

In [ ]:
def plot_combined_local_means_only() -> plt.Figure:
    """Plot only the regional mean NPVR-difference stars."""
    figure, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
    x = np.arange(len(THRESHOLDS), dtype=float)
    offsets = {"combined": -0.18, "pd": 0.0, "hc": 0.18}

    for axis, (metric, label) in zip(axes, LOCAL_METRICS.items()):
        group_means = {
            group: [
                regional_mean(npvr_difference[group][threshold][metric])
                for threshold in THRESHOLDS
            ]
            for group in offsets
        }
        all_means = np.concatenate([np.asarray(values) for values in group_means.values()])
        span = max(np.ptp(all_means), 0.20 * np.max(np.abs(all_means)), 1e-4)
        axis.set_ylim(np.min(all_means) - 0.20 * span, np.max(all_means) + 0.45 * span)

        for group, offset in offsets.items():
            means = group_means[group]
            axis.plot(
                x + offset, means, linestyle="--", linewidth=1.8, marker="*",
                markersize=14, color=COLORS[group], markeredgecolor="black",
            )
            for index, threshold in enumerate(THRESHOLDS):
                p_value = permutation_results[group][threshold][metric]["p_corrected"]
                axis.text(
                    x[index] + offset, means[index] + 0.10 * span,
                    significance_label(p_value), ha="center", va="bottom",
                    color=COLORS[group], fontsize=11, fontweight="bold",
                )

        axis.axhline(0, color="gray", linewidth=0.8)
        axis.set_title(label, fontweight="bold", fontsize=16)
        axis.set_ylabel("Mean NPVR difference", fontsize=14)
        axis.tick_params(axis="both", labelsize=14)
        axis.grid(axis="y", color="0.9", linewidth=0.8)

    axes[-1].set_xticks(x, [f"T = {threshold:g}" for threshold in THRESHOLDS])
    axes[-1].set_xlabel("Threshold Values", fontweight="bold", fontsize=14)
    axes[-1].text(
        0.5, -0.25, "* p<0.05; ** p<0.01; *** p<0.001; ns = not significant",
        transform=axes[-1].transAxes, ha="center", fontsize=10,
    )
    add_combined_caption(figure, 0.05)
    figure.tight_layout(rect=(0, 0.08, 1, 1))
    return figure


combined_local_means_only = plot_combined_local_means_only()
save_figure(combined_local_means_only, "fig5-localPD+HC-stars-only")
plt.show()

## 6. PD and HC local variability and NPVR differences

NV and PV use the left axis; regional NPVR differences and their means use the right axis.

In [ ]:
def plot_group_local(group: str, numerical_color: str, population_color: str) -> plt.Figure:
    figure, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
    x = np.arange(len(THRESHOLDS), dtype=float)
    for axis, (metric, label) in zip(axes, LOCAL_METRICS.items()):
        ratio_axis = axis.twinx()
        means = []
        for index, threshold in enumerate(THRESHOLDS):
            numerical = variability_difference[group][threshold]["numerical"][metric]
            population = variability_difference[group][threshold]["population"][metric]
            ratio = npvr_difference[group][threshold][metric]
            draw_box(axis, numerical, x[index] - 0.15, color=numerical_color, width=0.16)
            draw_box(axis, population, x[index] + 0.02, color=population_color, width=0.16)
            draw_box(ratio_axis, ratio, x[index] + 0.22, COLORS[group], width=0.16)
            mean = regional_mean(ratio)
            means.append(mean)
            ratio_axis.plot(x[index] + 0.22, mean, marker="*", markersize=11,
                            color=COLORS[group], markeredgecolor="black")
            p_value = permutation_results[group][threshold][metric]["p_corrected"]
            ratio_axis.text(x[index] + 0.22, max(finite_values(ratio)), significance_label(p_value),
                            ha="center", va="bottom", fontsize=10)
        ratio_axis.plot(x + 0.22, means, color=COLORS[group], linestyle="--", linewidth=1.8)
        axis.axhline(0, color="gray", linewidth=0.8)
        ratio_axis.axhline(0, color="gray", linewidth=0.5)
        axis.set_title(label, fontweight="bold", fontsize=16)
        axis.set_ylabel("Variability Differnces", fontsize=14)
        ratio_axis.set_ylabel("Mean NPVR", fontsize=14)
        axis.tick_params(axis="both", labelsize=14)
        ratio_axis.tick_params(axis="y", labelsize=14)
        axis.grid(axis="y", color="0.9", linewidth=0.8)
    axes[-1].set_xticks(x, [f"T = {t:g}" for t in THRESHOLDS])
    axes[-1].set_xlabel("Threshold Values", fontweight="bold", fontsize=14)
    axes[-1].text(0.5, -0.25, "* p<0.05; ** p<0.01; *** p<0.001; ns = not significant",
                  transform=axes[-1].transAxes, ha="center", fontsize=10)
    add_group_caption(figure, group, 0.05, numerical_color, population_color)
    figure.tight_layout(rect=(0, 0.08, 1, 1))
    return figure


pooled_local = plot_group_local("combined", numerical_color=COLORS["numerical"], population_color=COLORS["population"])
save_figure(pooled_local, "fig5-localpooledPD+HC")
plt.show()

pd_local = plot_group_local("pd", numerical_color="#FBA092", population_color="#770B07")
save_figure(pd_local, "fig5-localpd")
plt.show()

hc_local = plot_group_local("hc", numerical_color="#80ACF9", population_color="#031E4D")
save_figure(hc_local, "fig5-localhhc")
plt.show()

## 7. PD and HC global variability and NPVR differences

In [ ]:
def plot_group_global(group: str, numerical_color: str, population_color: str) -> plt.Figure:
    figure, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    x = np.arange(len(THRESHOLDS), dtype=float)
    for axis, (metric, label) in zip(axes, GLOBAL_METRICS.items()):
        numerical = [float(variability_difference[group][t]["numerical"][metric]) for t in THRESHOLDS]
        population = [float(variability_difference[group][t]["population"][metric]) for t in THRESHOLDS]
        ratio = [float(npvr_difference[group][t][metric]) for t in THRESHOLDS]
        axis.plot(x - 0.04, numerical, color=numerical_color, marker="o",
                  markeredgecolor="black", markersize=8, linestyle="None")
        axis.plot(x + 0.04, population, color=population_color, marker="o",
                  markeredgecolor="black", markersize=8, linestyle="None")
        ratio_axis = axis.twinx()
        ratio_axis.plot(x, ratio, color=COLORS[group], marker="*", markersize=11,
                        markeredgecolor="black", linestyle="--", linewidth=1.8)
        axis.axhline(0, color="gray", linewidth=0.8)
        ratio_axis.axhline(0, color="gray", linewidth=0.5)
        axis.set_title(label, fontweight="bold", fontsize=16)
        axis.set_ylabel("Variability (NV / PV)", fontsize=14)
        ratio_axis.set_ylabel("Mean NPVR", fontsize=14)
        axis.tick_params(axis="both", labelsize=14)
        ratio_axis.tick_params(axis="y", labelsize=14)
        axis.grid(axis="y", color="0.9", linewidth=0.8)
    axes[-1].set_xticks(x, [f"T = {t:g}" for t in THRESHOLDS])
    axes[-1].set_xlabel("Threshold", fontweight="bold", fontsize=14)
    add_group_caption(figure, group, 0.005, numerical_color, population_color)
    figure.tight_layout(rect=(0, 0.10, 1, 1))
    return figure


pooled_global = plot_group_global("combined", numerical_color=COLORS["numerical"], population_color=COLORS["population"])
save_figure(pooled_global, "fig5-globalpooledPD+HC")
plt.show()

pd_global = plot_group_global("pd", numerical_color="#FBA092", population_color="#770B07")
save_figure(pd_global, "fig5-globalhpd")
plt.show()

hc_global = plot_group_global("hc", numerical_color="#80ACF9", population_color="#031E4D")
save_figure(hc_global, "fig5-globalhhc")
plt.show()

### Parkinson's disease
![PD global variability and NPVR differences](Figures/fig5-globalhpd.png)

### Healthy controls
![HC global variability and NPVR differences](Figures/fig5-globalhhc.png)

## 8. Permutation-null diagnostics

The gray histograms show the sign-flip null distributions; the colored dashed line is the observed regional mean difference.

In [ ]:
def plot_permutation_diagnostics(group: str) -> plt.Figure:
    figure, axes = plt.subplots(4, 6, figsize=(18, 10), sharex="row", sharey="row")
    for row, (metric, metric_label) in enumerate(LOCAL_METRICS.items()):
        for column, threshold in enumerate(THRESHOLDS):
            axis = axes[row, column]
            result = permutation_results[group][threshold][metric]
            axis.hist(result["null"], bins=20, color="0.75", edgecolor="white")
            axis.axvline(result["statistic"], color=COLORS[group], linestyle="--", linewidth=1.8)
            axis.text(0.04, 0.90, f"p={result['p_corrected']:.3g}",
                      transform=axis.transAxes, fontsize=8, va="top")
            if row == 0: axis.set_title(f"T = {threshold:g}", fontweight="bold", fontsize=16)
            if column == 0: axis.set_ylabel(metric_label, fontsize=14)
            axis.tick_params(axis="both", labelsize=14)
            axis.grid(axis="y", color="0.9", linewidth=0.8)
    figure.suptitle(f"Permutation null distributions — {GROUP_LABELS[group]}", fontweight="bold", fontsize=16)
    figure.supxlabel("Permuted mean NPVR difference", fontsize=14)
    figure.tight_layout(rect=(0, 0.02, 1, 0.97))
    return figure


for group in ("combined", "pd", "hc"):
    diagnostic = plot_permutation_diagnostics(group)
    save_figure(diagnostic, f"permutation-null-{group}")
    plt.show()

![PD+HC permutation diagnostics](Figures/permutation-null-combined.png)

![PD permutation diagnostics](Figures/permutation-null-pd.png)

![HC permutation diagnostics](Figures/permutation-null-hc.png)

## Interpretation guide

- Positive $\Delta\mathrm{NPVR}$ means NPVR is larger without confound regression.
- Positive $\Delta\mathrm{NV}$ or $\Delta\mathrm{PV}$ means variability is larger with confound regression.
- Local-metric boxes summarize the 100 Schaefer regions; global metrics are scalar estimates.
- Corrected permutation labels describe evidence that the regional mean $\Delta\mathrm{NPVR}$ is below zero.